In [7]:
from dotenv import load_dotenv
load_dotenv()

import re
import polars as pl
from google import genai
from google.genai import types
from pathlib import Path
import base64
from scrapling.fetchers import FetcherSession
from aiofiles import open
from curl_cffi import AsyncSession
from typing import cast
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from enum import StrEnum
from datetime import datetime

## Crawling

In [2]:
with FetcherSession(impersonate='chrome') as session:
    page = session.get(
        'https://doe.gov.ph/articles/group/liquid-fuels?maincat=Retail%20Pump%20Prices&subcategory=NCR%20Pump%20Prices&display_type=Card'
    )

[2026-05-31 15:41:38] INFO: Fetched (200) <GET https://doe.gov.ph/articles/group/liquid-fuels?maincat=Retail%20Pump%20Prices&subcategory=NCR%20Pump%20Prices&display_type=Card> (referer: https://www.google.com/)


In [3]:
link_elements = (
    page
    .css('div.ex1')
    .css('a[href^="https://prod-cms.doe.gov.ph"]')
)
links = [l.attrib.get('href') for l in link_elements]
links

['https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-04282026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-05052026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-05122026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-05192026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-03312026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-04072026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-04142026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-04212026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-02242026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-03032026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-03102026n-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-031720

In [79]:
def func(x: str):
    partition = x.replace('https://prod-cms.doe.gov.ph/documents/d/', '')
    
    if match_ := re.search(r'^.+-(\d{8}).*?$', x):
        return datetime.strptime(match_.group(1), '%m%d%Y')
    
    if match_ := re.search(r'^.+-(\d{2}-\d{2}-\d{4}).+$', x):
        return datetime.strptime(match_.group(1), '%m-%d-%Y')
    
    if match_ := re.search(r'^.+_(\d{4}_\w+-\d{1,2}).+$', x):
        return datetime.strptime(match_.group(1), '%Y_%b-%d')
    
    if match_ := re.search(r'^.+_(\d{4}-\d{2}-\d{2}).*$', x):
        return datetime.strptime(match_.group(1), '%Y-%m-%d')
    
    if match_ := re.search(r'^.+[\-_](\d{4}-\w+-\d{1,2}).*$', x):
        try:
            return datetime.strptime(match_.group(1), '%Y-%b-%d')
        except:
            dat = match_.group(1).split('-')
            if len(dat[1]) == 4:
                dat[1] = dat[1][:3]
                return datetime.strptime('-'.join(dat), '%Y-%b-%d')
            return match_.group(1)
        
    if match_ := re.search(r'^.+_(\d{4}_\w{3}_\d{2}).*$', x):
        return datetime.strptime(match_.group(1), '%Y_%b_%d')
    
    if match_ := re.search(r'^.+_(\d{4}_\w+_\d{2}).*$', x):
        return datetime.strptime(match_.group(1), '%Y_%B_%d')
    
    return partition.split('/')[-1]

list(map(func, links))

[datetime.datetime(2026, 4, 28, 0, 0),
 datetime.datetime(2026, 5, 5, 0, 0),
 datetime.datetime(2026, 5, 12, 0, 0),
 datetime.datetime(2026, 5, 19, 0, 0),
 datetime.datetime(2026, 3, 31, 0, 0),
 datetime.datetime(2026, 4, 7, 0, 0),
 datetime.datetime(2026, 4, 14, 0, 0),
 datetime.datetime(2026, 4, 21, 0, 0),
 datetime.datetime(2026, 2, 24, 0, 0),
 datetime.datetime(2026, 3, 3, 0, 0),
 datetime.datetime(2026, 3, 10, 0, 0),
 datetime.datetime(2026, 3, 17, 0, 0),
 datetime.datetime(2026, 3, 24, 0, 0),
 datetime.datetime(2026, 2, 3, 0, 0),
 datetime.datetime(2026, 2, 10, 0, 0),
 datetime.datetime(2026, 2, 17, 0, 0),
 datetime.datetime(2026, 1, 6, 0, 0),
 datetime.datetime(2026, 1, 13, 0, 0),
 datetime.datetime(2026, 1, 20, 0, 0),
 datetime.datetime(2026, 1, 27, 0, 0),
 datetime.datetime(2025, 12, 2, 0, 0),
 datetime.datetime(2025, 12, 9, 0, 0),
 datetime.datetime(2025, 12, 16, 0, 0),
 datetime.datetime(2025, 12, 23, 0, 0),
 datetime.datetime(2025, 12, 30, 0, 0),
 datetime.datetime(2025, 10

In [5]:
async with AsyncSession() as session:
    session = cast(AsyncSession, session)
    filename = f"{links[0].split('/')[-1]}.pdf"
    res = cast(
        AsyncSession,
        await session.get(links[0], stream=True, impersonate='chrome')
    )
    async with open(f'./data/pdfs/{filename}', 'wb') as f:
        async for chunk in res.aiter_content(65536):
            await f.write(chunk)

## OCR

In [6]:
links[0]

'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-04282026-pdf'

In [ ]:
class FuelType(StrEnum):
    RON100 = 'RON100'
    RON97 = 'RON97'
    RON95 = 'RON95'
    RON91 = 'RON91'
    DIESEL = 'DIESEL'
    DIESEL_PLUS = 'DIESEL_PLUS'
    KEROSENE = 'KEROSENE'
    
class FuelBrand(StrEnum):
    PETRON = 'PETRON'
    SHELL = 'SHELL'
    CALTEX = 'CALTEX'
    PHOENIX = 'PHOENIX'
    TOTAL = 'TOTAL'
    FLYING_V = 'FLYING_V'
    UNIOIL = 'UNIOIL'
    SEAOIL = 'SEAOIL'
    PTT = 'PTT'
    INDEPENDENT = 'INDEPENDENT'


class FuelPrice(BaseModel):
    area: str = Field(description="Name of the area/city")
    product: FuelType
    brand: FuelBrand
    min_price: float | None = Field(None, ge=0.0, description="Minimum of the price range for a single combination of product and brand")
    max_price: float | None = Field(None, ge=0.0, description="Maximum of the price range for a single combination of product and brand")
    overall_range_min: float | None = Field(None, ge=0.0, description="Minimum price indicated under 'OVERALL RANGE' column")
    overall_range_max: float | None = Field(None, ge=0.0, description="Maximum price indicated under 'OVERALL RANGE' column")
    common_price: float | None = Field(None, ge=0.0, description="Price indicated under 'COMMON PRICE' column")
    
class StructuredOutput(BaseModel):
    results: list[FuelPrice]
    
gem = genai.Client()
system_prompt="""
You are an expert OCR Agent in the Philippine energy industry. Your role is to read
the provided PDF(s) containing fuel pump prices and return a structured output.
PDFs may contain a textual table or a screenshot/photocopy of a table.
"""

In [ ]:
filepath = Path(f'./data/pdfs/{filename}').resolve()

res = gem.models.generate_content(
    model="gemini-3.5-flash",
    contents=[
        types.Part.from_bytes(
            data=filepath.read_bytes(),
            mime_type='application/pdf',
        ),
        system_prompt,
    ],
    config=types.GenerateContentConfig(
          response_schema=StructuredOutput,
          response_mime_type='application/json',
    ),
)

In [37]:
pl.from_dicts(StructuredOutput.model_validate_json(res.text).results)

area,product,brand,min_price,max_price,overall_range_min,overall_range_max,common_price
str,str,str,f64,f64,f64,f64,f64
"""Caloocan City""","""RON97""","""SHELL""",102.5,103.0,79.79,103.0,null
"""Caloocan City""","""RON97""","""SEAOIL""",79.79,80.13,79.79,103.0,null
"""Caloocan City""","""RON95""","""PETRON""",78.6,83.3,71.9,96.6,93.9
"""Caloocan City""","""RON95""","""SHELL""",91.9,93.9,71.9,96.6,93.9
"""Caloocan City""","""RON95""","""CALTEX""",96.6,96.6,71.9,96.6,93.9
…,…,…,…,…,…,…,…
"""Navotas City""","""RON95""","""FLYING_V""",82.3,82.3,82.3,96.6,null
"""Navotas City""","""RON91""","""CALTEX""",89.8,89.8,81.3,89.8,null
"""Navotas City""","""RON91""","""FLYING_V""",81.3,81.3,81.3,89.8,null
